# 07. Academic Viva Voce Defense Guide & Theoretical Q&A
## B.Tech CSE 5th Semester - Machine Learning Innovative Assignment

This notebook compiles the complete theoretical justification, mathematical explanations, and expected questions from university evaluators with model answers.



---
### ❓ Question 1: Why did you choose HASYv2 instead of the standard MNIST dataset?
**Student Answer:**
- **Toy Dataset vs Real-World Scope:** MNIST only contains 10 Arabic digits ($0–9$), which is a saturated, closed-domain toy benchmark. Digit classification does not address the core challenges of scientific OCR.
- **Mathematical Complexity:** Real STEM applications (calculus, physics, linear algebra) require recognizing complex non-linear operators ($+, -, \times, \div, \pm$), relations ($<, >, \le, \ge, \ne$), calculus radicals ($\sqrt{}, \int, \sum, \prod, \infty, \partial$), and Greek variables ($\alpha, \beta, \pi, \theta, \lambda$).
- **Authentic Imbalance & Variations:** HASYv2 (Martin Thoma, Karlsruhe Institute of Technology, CC0 1.0) contains authentic human handwriting across 369 classes with genuine variations in stroke thickness, handwriting tilt, and frequency distribution.



---
### ❓ Question 2: Why did you invert image polarity (black background with white strokes)?
**Student Answer:**
- **Numerical Sparsity & Convolution Padding:** In raw paper scans, the background is white (value 255) and the ink is dark (value 0). In deep learning and CNNs, standard convolutional layers use zero-padding (padding with 0). If the background is 255, zero-padding introduces artificial rectangular high-contrast borders around the symbol, which distorts edge filters.
- Inverting the image so background is 0 and strokes are near 1.0 (or 255) ensures that zero-padding seamlessly matches the background, maximizing numerical sparsity and gradient stability.



---
### ❓ Question 3: How did you strictly guarantee that there is NO data leakage?
**Student Answer:**
1. **Splitting First:** The dataset was partitioned into 70% Train (26,809), 15% Validation (5,745), and 15% Test (5,745) using stratified sampling with a fixed random seed (`random_state=42`) *before* any feature extraction or scaling.
2. **Transform-Only Test Data:** The feature transformers (`StandardScaler`, HOG pipeline, class weight calculations) were fitted *exclusively* on the training split.
3. **Untouched Evaluation:** The test set was transformed using the training parameters and evaluated only at the very end. Test information never influenced hyperparameter tuning or model selection.



---
### ❓ Question 4: Why did HOG feature engineering boost accuracy from 81.03% to 92.90%?
**Student Answer:**
- **Raw Pixels (1,024-D):** Encode absolute photometric intensity at rigid $(x, y)$ coordinates. If a handwritten character is shifted by just 2 pixels or drawn with a thicker pen, raw pixel distance surges, leading to brittle linear boundaries.
- **HOG Descriptors (324-D):** Divide the image into $8 \times 8$ local cells and calculate gradient orientation histograms across 9 angular bins (Dalal-Triggs method). HOG captures the underlying geometric stroke directions invariant to minor translation shifts, illumination variations, and pen thickness changes.



---
### ❓ Question 5: Why did you report Balanced Accuracy and Macro F1 alongside standard Accuracy?
**Student Answer:**
- **Class Imbalance Sensitivity:** In HASYv2, the sample distribution is skewed (e.g., $\int$ has 3,514 samples while $+$ has only 90 samples).
- A naive classifier that blindly predicts majority classes could achieve $>85\%$ standard accuracy while having $0\%$ accuracy on minority symbols.
- **Balanced Accuracy:** Calculates the unweighted average recall across all 30 classes.
- **Macro F1:** Calculates the harmonic mean of precision and recall for each individual class and averages them equally, ensuring minority classes like $+$ and $>$ are not obscured.



---
### ❓ Question 6: What are the main confusion failure modes and their physical causes?
**Student Answer:**
1. **$\pi$ vs $\prod$ (43 test misclassifications):** Typographically, the capital Product symbol $\prod$ is an enlarged Pi with a top horizontal bar and two vertical legs. Without contextual scale, freehand $\pi$ and $\prod$ share nearly identical stroke geometry.
2. **$\pm$ vs $+$ (17 test misclassifications):** If the bottom horizontal minus stroke of $\pm$ is faint or detached, models naturally classify the remaining cross as $+$.
3. **$\le$ vs $<$ (12 test misclassifications):** The lower horizontal stroke in $\le$ is often drawn faintly or merged during thresholding.
4. **$\alpha$ vs $\infty$ (11 test misclassifications):** Both symbols consist of looping continuous curves; a closed right tail on $\alpha$ matches the topology of an infinity sign.



In [ ]:
# Quick inspection of test evaluation metrics
import json, pandas as pd
with open('../results/metrics/metrics_summary.json') as f:
    metrics = json.load(f)
pd.DataFrame(metrics).T[['accuracy', 'balanced_accuracy', 'macro_f1', 'inference_latency_ms', 'model_size_mb']]
